In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# FINAL STABLE DeepSeek LoRA TRAINING + EVALUATION (ERROR-FREE)
# ============================================================

# ================================
# INSTALL & ENV SETUP (RUN ONCE)
# ================================
import subprocess, sys, os
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade",
    "transformers", "datasets", "accelerate",
    "peft", "bitsandbytes", "pandas", "scikit-learn", "tqdm"
], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

# ================================
# IMPORTS
# ================================
import torch
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model

# ================================
# CONFIG
# ================================
MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"
DATA_PATH = "/kaggle/input/datasets/nikunjnawal009/deepseek-devignx26"
OUTPUT_DIR = "./deepseek-lora-final"

MAX_LEN = 512
EPOCHS = 4
BATCH_SIZE = 2
THRESHOLD = 0.40

# ================================
# LOAD DATA
# ================================
file = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(DATA_PATH, file))

df = df[["code", "label"]].dropna()
df = df[df["label"].isin([0, 1])]
df = df.rename(columns={"code": "text"}).reset_index(drop=True)

# ================================
# 🔥 BALANCE DATA (CRITICAL FIX)
# ================================
vuln_df = df[df["label"] == 1]
safe_df = df[df["label"] == 0]

df = pd.concat([safe_df, vuln_df, vuln_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# Split
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# ================================
# TOKENIZER
# ================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# ================================
# FORMAT + TOKENIZE
# ================================
def format_and_tokenize(example):
    prompt = f"""You are a vulnerability classifier.

Classify code:
0 = Safe
1 = Vulnerable

Only output 1 if code clearly contains:
- buffer overflow
- unsafe memory operations
- pointer misuse

Code:
{example['text']}

Output:
"""
    full_text = prompt + str(example["label"])

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = train_dataset.map(format_and_tokenize)
val_dataset = val_dataset.map(format_and_tokenize)

# ================================
# MODEL (4-BIT QUANTIZATION)
# ================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ================================
# LoRA CONFIG
# ================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ================================
# TRAINER (STABLE)
# ================================
class StableTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

# ================================
# TRAINING
# ================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)

trainer = StableTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print("\n🚀 Training Started...\n")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n✅ Training Complete\n")

# ================================
# 🔥 CALIBRATED EVALUATION
# ================================
def evaluate_model(df, model, tokenizer):
    model.eval()

    id0 = tokenizer("0", add_special_tokens=False)["input_ids"][0]
    id1 = tokenizer("1", add_special_tokens=False)["input_ids"][0]

    preds = []
    texts = df["text"].tolist()

    for code in tqdm(texts, desc="Evaluating"):
        prompt = f"""You are a vulnerability classifier.

Code:
{code}

Output:
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1]

        probs = torch.softmax(torch.tensor([logits[id0], logits[id1]]), dim=0)
        p1 = probs[1].item()

        pred = 1 if p1 > THRESHOLD else 0
        preds.append(pred)

    y_true = df["label"]

    print("\n===== FINAL RESULTS =====")
    print(f"Accuracy : {accuracy_score(y_true, preds):.4f}")
    print(f"Precision: {precision_score(y_true, preds, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, preds, zero_division=0):.4f}")
    print(f"F1 Score : {f1_score(y_true, preds, zero_division=0):.4f}\n")

    print(classification_report(y_true, preds, zero_division=0))

# ================================
# RUN EVALUATION
# ================================
evaluate_model(val_df, model, tokenizer)

Map:   0%|          | 0/3527 [00:00<?, ? examples/s]

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🚀 Training Started...



Step,Training Loss
50,25.065967
100,24.963257
150,23.463210
